In [1]:
%cd ../../
%load_ext dotenv
%dotenv

/Users/hoangle/Projects/untangling-people/ylva/fwo_models


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
from darts.models.forecasting.catboost_model import CatBoostModel
from darts import TimeSeries

/Users/hoangle/Projects/untangling-people/ylva/fwo_models/.venv/lib/python3.11/site-packages/fs/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)  # type: ignore


# Read file

In [3]:
PATH_DIR_FORECASTED = Path("data/processed/waste_daily/forecasted")

In [4]:
path = "data/processed/dim_holidays_uhelsinki.xlsx"
dim_holiday = (
    pl.read_excel(path)
    .filter(pl.col('date').dt.weekday() <= 5)
)


In [5]:
path = "data/processed/dim_exams_uhelsinki.xlsx"
dim_exam = (
     pl.read_excel(path)
    .filter(pl.col('date').dt.weekday() <= 5)
)

dim_exam.head()

date,is_exam
date,bool
2023-01-02,false
2023-01-03,false
2023-01-04,false
2023-01-05,false
2023-01-06,false


In [6]:
path_historical = Path("data/processed/waste.parquet")

waste_daily_raw = pl.read_parquet(path_historical)

waste_daily_raw.head()

id,date,restaurant,waste,src
str,date,i64,f32,str
"""2024-11-01|3""",2024-11-01,3,19.0,"""Data Physicum 01112024-3103202…"
"""2024-11-04|3""",2024-11-04,3,21.0,"""Data Physicum 01112024-3103202…"
"""2024-11-05|3""",2024-11-05,3,15.0,"""Data Physicum 01112024-3103202…"
"""2024-11-06|3""",2024-11-06,3,15.0,"""Data Physicum 01112024-3103202…"
"""2024-11-07|3""",2024-11-07,3,15.0,"""Data Physicum 01112024-3103202…"


In [7]:
restaurant_id = 4
waste_daily = (
    pl.DataFrame({
        'date': pl.Series(pd.date_range(waste_daily_raw['date'].min(), waste_daily_raw['date'].max(), freq='b')).dt.date()
    })
    .join(
        waste_daily_raw.filter(pl.col('restaurant') == restaurant_id),
        on='date', how='left'
    )
    .with_columns(
        (pl.col('waste').is_null() | (pl.col('waste') == 0)).alias('is_holiday'),
        pl.col('waste').fill_null(0)
    )
    .join(dim_exam, on='date', how='left')
    .with_columns(pl.col('is_exam').fill_null(False))


    .with_columns(
        pl.col('date').dt.weekday().alias('weekday'),
        pl.col('date').dt.day().alias('day'),
        pl.col('date').dt.week().alias('week'),
        pl.col('date').dt.month().alias('month'),
        pl.col('date').dt.year().alias('year'),
    )
    .with_columns(
        (pl.col('weekday') * 2 * np.pi / 7).sin().alias('weekday_sin'),
        (pl.col('weekday') * 2 * np.pi / 7).cos().alias('weekday_cos'),
        (pl.col('day') * 2 * np.pi / 31).sin().alias('day_sin'),
        (pl.col('day') * 2 * np.pi / 31).cos().alias('day_cos'),
        (pl.col('week') * 2 * np.pi / 53).sin().alias('week_sin'),
        (pl.col('week') * 2 * np.pi / 53).cos().alias('week_cos'),
        (pl.col('month') * 2 * np.pi / 12).sin().alias('month_sin'),
        (pl.col('month') * 2 * np.pi / 12).cos().alias('month_cos'),
    )
    .fill_null(0.)
    .fill_nan(0.)

    .with_row_index()

    .to_pandas()
)
waste_daily.head()

,index,date,id,restaurant,waste,src,is_holiday,is_exam,weekday,day,...,month,year,weekday_sin,weekday_cos,day_sin,day_cos,week_sin,week_cos,month_sin,month_cos
0,0,2023-01-02,None,0.0,0.0,None,True,False,1.0,2.0,...,1.0,2023.0,0.781831,0.623490,0.394356,0.918958,0.118273,0.992981,0.5,0.866025
1,1,2023-01-03,None,0.0,0.0,None,True,False,2.0,3.0,...,1.0,2023.0,0.974928,-0.222521,0.571268,0.820763,0.118273,0.992981,0.5,0.866025
2,2,2023-01-04,None,0.0,0.0,None,True,False,3.0,4.0,...,1.0,2023.0,0.433884,-0.900969,0.724793,0.688967,0.118273,0.992981,0.5,0.866025
3,3,2023-01-05,None,0.0,0.0,None,True,False,4.0,5.0,...,1.0,2023.0,-0.433884,-0.900969,0.848644,0.528964,0.118273,0.992981,0.5,0.866025
4,4,2023-01-06,None,0.0,0.0,None,True,False,5.0,6.0,...,1.0,2023.0,-0.974928,-0.222521,0.937752,0.347305,0.118273,0.992981,0.5,0.866025


# Train and forecast

In [8]:
cols_tgt = [
    'waste'
]
cols_cov = [
    'weekday_sin',
    'weekday_cos',
    'day_sin',
    'day_cos',
    'week_sin',
    'week_cos',
    'month_sin',
    'month_cos',
    'is_holiday',
    'is_exam'
]

ts_target = TimeSeries.from_dataframe(
    df=waste_daily,
    time_col='index',
    fillna_value=0.,
    value_cols=cols_tgt,
)

ts_cov = TimeSeries.from_dataframe(
    df=waste_daily,
    time_col='index',
    value_cols=cols_cov,
)

In [9]:
lags = 5
output_chunk_length = 1

model = CatBoostModel(lags=lags, output_chunk_length=output_chunk_length, lags_future_covariates=[0])
model.fit(ts_target, future_covariates=ts_cov)

CatBoostModel(lags=5, lags_past_covariates=None, lags_future_covariates=[0], output_chunk_length=1, output_chunk_shift=0, add_encoders=None, likelihood=None, quantiles=None, random_state=None, multi_models=True, use_static_covariates=True, categorical_past_covariates=None, categorical_future_covariates=None, categorical_static_covariates=None)

### Forecast

In [10]:
date_forecast_start = '2025-11-01'
date_forecast_end = '2026-05-31'

In [11]:
cov_holiday_future = (
    pl.DataFrame({
        'date': pl.Series(pd.date_range(date_forecast_start, date_forecast_end, freq='b')).dt.date()
    })

    .join(dim_holiday, on='date', how='left')
    .with_columns(pl.col('is_holiday').fill_null(False))

    .join(dim_exam, on='date', how='left')
    .with_columns(pl.col('is_exam').fill_null(False))

    .with_columns(
        pl.col('date').dt.weekday().alias('weekday'),
        pl.col('date').dt.day().alias('day'),
        pl.col('date').dt.week().alias('week'),
        pl.col('date').dt.month().alias('month'),
        pl.col('date').dt.year().alias('year'),
    )
    .with_columns(
        (pl.col('weekday') * 2 * np.pi / 7).sin().alias('weekday_sin'),
        (pl.col('weekday') * 2 * np.pi / 7).cos().alias('weekday_cos'),
        (pl.col('day') * 2 * np.pi / 31).sin().alias('day_sin'),
        (pl.col('day') * 2 * np.pi / 31).cos().alias('day_cos'),
        (pl.col('week') * 2 * np.pi / 53).sin().alias('week_sin'),
        (pl.col('week') * 2 * np.pi / 53).cos().alias('week_cos'),
        (pl.col('month') * 2 * np.pi / 12).sin().alias('month_sin'),
        (pl.col('month') * 2 * np.pi / 12).cos().alias('month_cos'),
    )

    .with_columns(
        pl.Series("index", range(waste_daily['index'].max() + 1, waste_daily['index'].max() + 151))
    )
)

ts_cov_future = TimeSeries.from_dataframe(
    df=cov_holiday_future,
    time_col='index',
    value_cols=cols_cov,
)

In [12]:
predictions_raw = model.predict(
    len(ts_cov_future),
    ts_target,
    future_covariates=ts_cov_future
).to_dataframe()

predictions_raw.head()

,waste
index,
586,26.181361
587,32.493155
588,28.987045
589,27.307353
590,31.899372


In [13]:
predictions = (
    pl.concat(
        [cov_holiday_future[['date']], pl.from_pandas(predictions_raw.reset_index())],
        how="horizontal"
    )
    .drop('index')
    .with_columns(
        *[pl.col(col).clip(0) for col in cols_tgt]
    )
)

predictions.head()

date,waste
date,f64
2025-11-03,26.181361
2025-11-04,32.493155
2025-11-05,28.987045
2025-11-06,27.307353
2025-11-07,31.899372


Add other columns as fact table
    

In [14]:
cols_fact = [
    'date',
    'restaurant_id',
    'waste',
]

predictions = (
    predictions
    .with_columns(
        pl.lit(restaurant_id).cast(pl.Int64).alias('restaurant_id'),
    )

    .join(dim_holiday, on='date', how='left')
    .with_columns(
        pl.when(pl.col('is_holiday')).then(0).otherwise(pl.col('waste')).alias('waste')
    )

    .select(cols_fact)
)


Save

In [15]:
path_forecasted = PATH_DIR_FORECASTED / f"{restaurant_id}.xlsx"
path_forecasted.parent.mkdir(exist_ok=True, parents=True)

predictions.write_excel(path_forecasted)